# MergeSortBottomUp (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_python.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `MergeSortBottomUp`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/MergeSortBottomUp_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fib_rec` (snake_case).

## MergeSort ricorsivo e MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: un array $a[1..n]$ di naturali. *Output*: l'array ordinato in modo non decrescente. La stessa suddivisione di Merge Sort, ma **senza ricorsione**: prima si fondono coppie di sotto-array di lunghezza $1$, poi di lunghezza $2$, $4$, … con la procedura `Merge` del capitolo 17 (qui ripetuta). Esegue $\lceil\log_2 n\rceil$ passate e $\Theta(n\log_2 n)$ passi, come la versione ricorsiva, e non usa la pila delle chiamate.

Prima l'esempio del capitolo $a=[5,2,4,6,1,3]$, passata per passata (gli indici sono da 0). Poi il confronto, su array pseudo-casuali (stesso generatore in tutti i linguaggi, seme $12345$, valori fra $1$ e $10^6$), del numero di **confronti** fra elementi di `InsertionSort`, di `MergeSort` e di `MergeSortBottomUp`: $\Theta(n^2)$ contro $\Theta(n\log_2 n)$, con le due versioni di Merge Sort praticamente uguali ("OK" indica che i tre algoritmi danno lo stesso array ordinato), e i tempi misurati.

In [1]:
import time

comparisons = 0   # confronti fra elementi

def merge(a, l, m, r):
    """Merge(a, l, m, r) del capitolo 17."""
    global comparisons
    n1 = m - l + 1
    n2 = r - m
    L = [0] * n1
    R = [0] * n2
    for i in range(n1):
        L[i] = a[l + i]
    for j in range(n2):
        R[j] = a[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:
        comparisons += 1
        if L[i] <= R[j]:
            a[k] = L[i]
            i += 1
        else:
            a[k] = R[j]
            j += 1
        k += 1
    while i < n1:
        a[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        a[k] = R[j]
        j += 1
        k += 1

def merge_sort(a, l, r):
    if l < r:
        m = l + (r - l) // 2
        merge_sort(a, l, m)
        merge_sort(a, m + 1, r)
        merge(a, l, m, r)

def merge_sort_bottom_up(a, n):
    """MergeSortBottomUp(a, n): ordina a[0..n-1] senza ricorsione."""
    width = 1
    while width < n:
        l = 0
        while l + width < n:                    # pseudocodice: l + width <= n
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            l = l + 2 * width
        width = 2 * width

def merge_sort_bottom_up_trace(a, n):
    width = 1
    passes = 0
    while width < n:
        l = 0
        parts = []
        while l + width < n:
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            parts.append(f"a[{l}..{m}] con a[{m + 1}..{r}]")
            l = l + 2 * width
        passes += 1
        print(f"width={width}: merge " + ", ".join(parts) + f" => {a}")
        width = 2 * width
    print(f"passate = {passes}")

def insertion_sort(a, n):
    """InsertionSort(a, n) (capitolo 3), con il conteggio dei confronti."""
    global comparisons
    for j in range(1, n):
        key = a[j]
        i = j - 1
        while i >= 0:
            comparisons += 1
            if a[i] > key:
                a[i + 1] = a[i]
                i = i - 1
            else:
                break
        a[i + 1] = key

def random_array(n, seed=12345):
    """Generatore congruenziale lineare: stessi numeri in Python, C++ e Java."""
    x = seed
    a = []
    for _ in range(n):
        x = (x * 1103515245 + 12345) % 2147483648
        a.append(1 + x % 1000000)
    return a

a = [5, 2, 4, 6, 1, 3]
print("a =", a, "  n =", len(a))
merge_sort_bottom_up_trace(a, len(a))

print()
print(f"{'n':>6s} {'InsertionSort':>14s} {'MergeSort':>10s} {'BottomUp':>9s}  confronti")
times = []
for n in [1000, 2000, 4000]:
    base = random_array(n)
    x1, x2, x3 = base[:], base[:], base[:]
    comparisons = 0
    t0 = time.perf_counter(); insertion_sort(x1, n); t1 = (time.perf_counter() - t0) * 1000
    c1 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort(x2, 0, n - 1); t2 = (time.perf_counter() - t0) * 1000
    c2 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort_bottom_up(x3, n); t3 = (time.perf_counter() - t0) * 1000
    c3 = comparisons
    ok = "OK" if (x1 == x2 == x3 == sorted(base)) else "ERROR"
    print(f"{n:>6d} {c1:>14d} {c2:>10d} {c3:>9d}  {ok}")
    times.append((n, t1, t2, t3))
print()
print("Tempi misurati (dipendono dalla macchina):")
for n, t1, t2, t3 in times:
    print(f"n={n:<6d} InsertionSort {t1:>10.3f} ms   MergeSort {t2:>8.3f} ms   MergeSortBottomUp {t3:>8.3f} ms")

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] con a[1..1], a[2..2] con a[3..3], a[4..4] con a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] con a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] con a[4..5] => [1, 2, 3, 4, 5, 6]
passate = 3

     n  InsertionSort  MergeSort  BottomUp  confronti
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Tempi misurati (dipendono dalla macchina):
n=1000   InsertionSort     25.161 ms   MergeSort    2.482 ms   MergeSortBottomUp    2.203 ms
n=2000   InsertionSort    105.176 ms   MergeSort    4.776 ms   MergeSortBottomUp    4.988 ms
n=4000   InsertionSort    436.729 ms   MergeSort   10.723 ms   MergeSortBottomUp   10.794 ms


**Prova tu.** Cambia $n$ (con $n=8000$ i confronti di `InsertionSort` quadruplicano rispetto a $n=4000$, quelli di Merge Sort poco più del doppio).

In [2]:
# Prova con n = 8000: i confronti di InsertionSort quadruplicano rispetto a n = 4000, quelli di Merge Sort poco più del doppio.
n = 8000
base = random_array(n)
x1, x2 = base[:], base[:]
comparisons = 0
insertion_sort(x1, n)
c1 = comparisons
comparisons = 0
merge_sort_bottom_up(x2, n)
print(c1, comparisons, x1 == x2)

16115652 93756 True


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).